# 🧱 XỬ LÝ DỮ LIỆU TRƯỚC KHI TRAIN (Data Preparation)

**Cấu phần (ii)** · Dự án Competitor Fare Forecasting (GSM)

| | |
|---|---|
| **Input** | `data/dataset_clean.parquet` (637.322 chuyến đã làm sạch) |
| **Output** | `snapshot_price_{15,60}min.parquet`, `snapshot_surge_{15,60}min.parquet` |
| **Tiếp theo** | mở `model/model_train.ipynb` để huấn luyện |

Notebook này làm **3 việc**:

1. **Gộp SNAPSHOT** theo 2 độ phân giải khác nhau cho 2 target
   - `price`: series = `source × destination × name` (cấp cuốc, tách từng hãng)
   - `surge`: series = `source × destination` (cấp thị trường, chỉ Lyft, bỏ Shared)
2. **Feature engineering** chỉ từ **QUÁ KHỨ** (chống rò rỉ): lag1/2/3, roll_mean3/6, roll_std3, `observation_age`
3. **Gán `data_split`** theo **THỜI GIAN**: train ≤10/12 · calibration 13–15/12 · test ≥16/12

---
## Mục lục
1. Cấu hình & nạp dữ liệu
2. Hàm gán split theo thời gian
3. Hàm feature thời gian
4. Hàm lag / rolling / độ trễ (chống rò rỉ)
5. Dựng bảng GIÁ (cấp cuốc)
6. Dựng bảng SURGE (cấp thị trường)
7. Chạy sinh 4 file snapshot
8. Kiểm tra kết quả

---
# 1️⃣ Cấu hình & nạp dữ liệu

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("../data")
IN   = DATA / "dataset_clean.parquet"

# --- Chia theo THOI GIAN (khong bao gio chia ngau nhien) ---
TRAIN_END              = "2018-12-10"
CALIB_START, CALIB_END = "2018-12-13", "2018-12-15"
TEST_START             = "2018-12-16"
# Luu y: 11-12/12 la 2 ngay BUFFER (bi loai) de train khong dinh sat calibration.

# Thoi tiet giu lai (bo cac cot proxy ngay: moonPhase, pressure, ozone)
WEATHER = ["temperature", "precipIntensity", "precipProbability",
           "humidity", "windSpeed", "visibility", "cloudCover"]

df = pd.read_parquet(IN)
print(f"Nap {len(df):,} chuyen x {df.shape[1]} cot tu {IN.name}")
print(f"Thoi gian: {df.local.min()}  ->  {df.local.max()}")
display(df[["local","cab_type","source","destination","name","distance","price","surge_multiplier"]].head())

Nap 637,322 chuyen x 63 cot tu dataset_clean.parquet
Thoi gian: 2018-11-25 22:40:46  ->  2018-12-18 14:15:11


,local,cab_type,source,destination,name,distance,price,surge_multiplier
0,2018-12-16 04:30:07.890000105,Lyft,Haymarket Square,North Station,Shared,0.44,5.0,1.0
1,2018-11-26 21:00:23.677000046,Lyft,Haymarket Square,North Station,Lux,0.44,11.0,1.0
2,2018-11-27 20:00:22.197999954,Lyft,Haymarket Square,North Station,Lyft,0.44,7.0,1.0
3,2018-11-29 23:53:02.749000072,Lyft,Haymarket Square,North Station,Lux Black XL,0.44,26.0,1.0
4,2018-11-28 22:49:20.223000050,Lyft,Haymarket Square,North Station,Lyft XL,0.44,9.0,1.0


---
# 2️⃣ Hàm gán split theo thời gian

Gán mỗi dòng vào `train` / `calibration` / `test` theo **ngày** của snapshot. Ngày rơi vào
buffer (11–12/12) hoặc ngoài khoảng → `drop`.

In [2]:
def gan_split(ngay: pd.Series) -> pd.Series:
    s = pd.Series("drop", index=ngay.index, dtype=object)
    s[ngay <= TRAIN_END] = "train"
    s[(ngay >= CALIB_START) & (ngay <= CALIB_END)] = "calibration"
    s[ngay >= TEST_START] = "test"
    return s

# Thu nghiem nhanh
_demo = pd.to_datetime(["2018-11-30","2018-12-11","2018-12-14","2018-12-17"])
print(dict(zip([str(d.date()) for d in _demo], gan_split(pd.Series(_demo)))))

{'2018-11-30': 'train', '2018-12-11': 'drop', '2018-12-14': 'calibration', '2018-12-17': 'test'}


---
# 3️⃣ Hàm feature thời gian

Từ mốc `snapshot` suy ra giờ / thứ / cuối tuần. Thêm `hour_sin/cos` (mã hoá chu kỳ cho giờ,
vì giờ **không đơn điệu** — kết luận từ báo cáo Study Relation).

In [3]:
def them_thoi_gian(g: pd.DataFrame) -> pd.DataFrame:
    t = g["snapshot"]
    g["hour_local"]    = t.dt.hour
    g["weekday_local"] = t.dt.weekday
    g["is_weekend"]    = (t.dt.weekday >= 5).astype(int)
    g["hour_sin"]      = np.sin(2 * np.pi * g.hour_local / 24)
    g["hour_cos"]      = np.cos(2 * np.pi * g.hour_local / 24)
    return g

---
# 4️⃣ Hàm lag / rolling / độ trễ — ⭐ CHỐNG RÒ RỈ

Đây là hàm quan trọng nhất. Với mỗi series (sắp theo thời gian), sinh:

| Feature | Ý nghĩa |
|---|---|
| `lag1/2/3` | Giá trị ở 1/2/3 mốc **trước** (dùng `shift`, không dùng mốc hiện tại) |
| `delta_1_2` | Xu hướng: đang tăng hay giảm |
| `roll_mean3/6` | Mức nền gần đây (tính trên các mốc **trước**) |
| `roll_std3` | Độ biến động |
| `observation_age_minutes` | **Dữ liệu đang cũ bao nhiêu phút** = khoảng cách tới snapshot trước |
| `observation_age_bucket` | Nhóm độ trễ (≤15p, 15–30p, 30–60p, 1–3h, >3h) |

> 🔒 **Mọi feature chỉ dùng `shift(k≥1)`** → tuyệt đối không nhìn thấy giá trị tại mốc hiện tại.

In [4]:
def them_lag(g: pd.DataFrame, keys: list, cot_dich: str, tien_to: str) -> pd.DataFrame:
    """Lag + rolling + do tre quan sat. CHI dung qua khu -> khong ro ri."""
    g = g.sort_values(keys + ["snapshot"]).reset_index(drop=True)
    gr = g.groupby(keys, observed=True)

    for k in (1, 2, 3):
        g[f"lag{k}_{tien_to}"] = gr[cot_dich].shift(k)
    g[f"delta_{tien_to}_1_2"] = g[f"lag1_{tien_to}"] - g[f"lag2_{tien_to}"]

    truoc = gr[cot_dich].shift(1)               # <-- luon lay tu QUA KHU
    idx = [g[k] for k in keys]
    for w in (3, 6):
        g[f"roll_mean{w}_{tien_to}"] = (truoc.groupby(idx)
                                        .rolling(w, min_periods=1).mean()
                                        .reset_index(drop=True))
    g[f"roll_std3_{tien_to}"] = (truoc.groupby(idx)
                                 .rolling(3, min_periods=2).std()
                                 .reset_index(drop=True))

    g["lag1_quote_count"] = gr["quote_count"].shift(1)
    g["so_quan_sat_truoc"] = gr.cumcount()

    # Do tre = snapshot nay cach snapshot truoc bao nhieu phut (= observation age)
    truoc_t = gr["snapshot"].shift(1)
    g["observation_age_minutes"] = (g["snapshot"] - truoc_t).dt.total_seconds() / 60
    g["observation_age_bucket"] = pd.cut(
        g["observation_age_minutes"], [-1, 15, 30, 60, 180, 1e9],
        labels=["<=15p", "15-30p", "30-60p", "1-3h", ">3h"]).astype(str)
    return g

---
# 5️⃣ Dựng bảng GIÁ (cấp cuốc)

`series = cab_type × source × destination × name` — mỗi loại dịch vụ có giá riêng nên **giữ
`name`** trong định nghĩa chuỗi. Target = **median** giá trong bucket.

In [5]:
def dung_bang_price(df: pd.DataFrame) -> pd.DataFrame:
    keys = ["cab_type", "source", "destination", "name"]
    agg = {"price": [("target_price", "median"),
                     ("price_min", "min"), ("price_max", "max")],
           "distance": [("distance_median", "median")],
           "id": [("quote_count", "size")]}
    gb = df.groupby(keys + ["snapshot"], observed=True)
    parts = [gb[c].agg(how).rename(name) for c, specs in agg.items() for name, how in specs]
    parts += [gb[w].median().rename(w) for w in WEATHER if w in df.columns]
    parts.append(gb["short_summary"].agg(
        lambda s: s.mode().iat[0] if len(s.mode()) else "NA").rename("short_summary"))
    g = pd.concat(parts, axis=1).reset_index()
    g["price_spread"] = g.price_max - g.price_min
    g = them_thoi_gian(g)
    g = them_lag(g, keys, "target_price", "price")
    return g

print("Ham dung_bang_price da san sang.")

Ham dung_bang_price da san sang.


---
# 6️⃣ Dựng bảng SURGE (cấp thị trường)

`series = source × destination` (KHÔNG có `name`), chỉ **Lyft**, bỏ **Shared** (không bao giờ surge).

> 💡 Dùng **`mean`** của `surge_multiplier`, KHÔNG dùng median: trong 1 bucket nếu chỉ 3/11 báo
> giá có surge thì median vẫn = 1.0 → mất tín hiệu. Mean giữ được cường độ.

In [6]:
def dung_bang_surge(df: pd.DataFrame) -> pd.DataFrame:
    d = df[(df.cab_type == "Lyft") & (df.name != "Shared")]
    keys = ["source", "destination"]
    gb = d.groupby(keys + ["snapshot"], observed=True)
    parts = [gb["surge_multiplier"].mean().rename("target_surge"),
             gb["is_surge"].mean().rename("target_surge_rate"),
             gb["surge_multiplier"].max().rename("surge_max"),
             gb["distance"].median().rename("distance_median"),
             gb["id"].size().rename("quote_count")]
    parts += [gb[w].median().rename(w) for w in WEATHER if w in d.columns]
    parts.append(gb["short_summary"].agg(
        lambda s: s.mode().iat[0] if len(s.mode()) else "NA").rename("short_summary"))
    g = pd.concat(parts, axis=1).reset_index()
    g["target_is_surge"] = (g.target_surge > 1.0).astype(int)   # nhi phan suy tu target_surge
    g = them_thoi_gian(g)
    g = them_lag(g, keys, "target_surge", "surge")
    truoc = g.groupby(keys, observed=True)["target_surge_rate"].shift(1)
    g["lag1_surge_rate"] = truoc
    g["roll_surge_rate6"] = (truoc.groupby([g[k] for k in keys])
                             .rolling(6, min_periods=1).mean()
                             .reset_index(drop=True))
    return g

print("Ham dung_bang_surge da san sang.")

Ham dung_bang_surge da san sang.


---
# 7️⃣ Chạy sinh 4 file snapshot

2 độ phân giải × 2 target = 4 file. **`15min` là bản chính** cho mục ii (nghiên cứu độ trễ);
`60min` giữ làm đối chứng. Đổi bucket chỉ cần đổi `dt.floor(freq)`.

In [7]:
BUCKETS = {"15min": "15min", "60min": "h"}
ket_qua = {}

for ten_bucket, freq in BUCKETS.items():
    df["snapshot"] = df["local"].dt.floor(freq)
    print("#"*60); print(f"  BUCKET {ten_bucket}  (floor '{freq}')"); print("#"*60)

    for ten, ham, base in [("PRICE", dung_bang_price, "snapshot_price"),
                           ("SURGE", dung_bang_surge, "snapshot_surge")]:
        t = time.time()
        g = ham(df)
        g["data_split"] = gan_split(g["snapshot"].dt.normalize())
        g = g[g.data_split != "drop"].reset_index(drop=True)

        fn = f"{base}_{ten_bucket}.csv"
        g.to_csv(DATA / fn, index=False, encoding="utf-8-sig")
        ket_qua[fn] = g

        keys = (["cab_type","source","destination","name"] if ten == "PRICE"
                else ["source","destination"])
        age = g["observation_age_minutes"]
        print(f"\n=== {ten} -> {fn} ===")
        print(f"  {len(g):,} snapshot x {g.shape[1]} cot  ({time.time()-t:.1f}s)")
        print(f"  So series: {g.groupby(keys, observed=True).ngroups:,}")
        print(f"  Split: {g.data_split.value_counts().to_dict()}")
        print(f"  Do tre giua 2 snapshot: median={age.median():.0f}p | p90={age.quantile(.9):.0f}p")
        if ten == "SURGE":
            print(f"  Ty le surge (mean>1): {g.target_is_surge.mean()*100:.2f}%")
        print(f"  Thieu lag1: {g.filter(like='lag1_').isna().mean().mean()*100:.1f}%")

print("\n>>> DA SINH XONG 4 FILE SNAPSHOT.")

############################################################
  BUCKET 15min  (floor '15min')
############################################################

=== PRICE -> snapshot_price_15min.csv ===
  471,649 snapshot x 36 cot  (48.2s)
  So series: 864
  Split: {'train': 289271, 'calibration': 97697, 'test': 84681}
  Do tre giua 2 snapshot: median=30p | p90=75p
  Thieu lag1: 0.2%

=== SURGE -> snapshot_surge_15min.csv ===
  66,445 snapshot x 36 cot  (6.6s)
  So series: 72
  Split: {'train': 39761, 'calibration': 14330, 'test': 12354}
  Do tre giua 2 snapshot: median=15p | p90=30p
  Ty le surge (mean>1): 14.45%
  Thieu lag1: 0.1%
############################################################
  BUCKET 60min  (floor 'h')
############################################################

=== PRICE -> snapshot_price_60min.csv ===
  245,596 snapshot x 36 cot  (23.8s)
  So series: 864
  Split: {'train': 145152, 'calibration': 53818, 'test': 46626}
  Do tre giua 2 snapshot: median=60p | p90=120p
  Thie

---
# 8️⃣ Kiểm tra kết quả

Xem thử bảng chính (`snapshot_price_15min`) và **kiểm chống rò rỉ trong từng split**.

In [8]:
gp = ket_qua["snapshot_price_15min.csv"]
print("BANG GIA 15min — vai dong dau (mot series):")
cols = ["snapshot","cab_type","source","name","target_price",
        "lag1_price","lag2_price","roll_mean6_price","observation_age_minutes","data_split"]
display(gp[gp.name=="Lyft"].sort_values(["source","destination","snapshot"])[cols].head(8))

BANG GIA 15min — vai dong dau (mot series):


,snapshot,cab_type,source,name,target_price,lag1_price,lag2_price,roll_mean6_price,observation_age_minutes,data_split
1572,2018-11-26 00:00:00,Lyft,Back Bay,Lyft,9.00,NaN,NaN,NaN,NaN,train
1573,2018-11-26 01:15:00,Lyft,Back Bay,Lyft,7.00,9.00,NaN,9.000000,75.0,train
1574,2018-11-26 04:45:00,Lyft,Back Bay,Lyft,7.00,7.00,9.00,8.000000,210.0,train
1575,2018-11-26 05:00:00,Lyft,Back Bay,Lyft,7.00,7.00,7.00,7.666667,15.0,train
1576,2018-11-26 05:15:00,Lyft,Back Bay,Lyft,7.00,7.00,7.00,7.500000,15.0,train
1577,2018-11-26 08:00:00,Lyft,Back Bay,Lyft,8.75,7.00,7.00,7.400000,165.0,train
1578,2018-11-26 08:45:00,Lyft,Back Bay,Lyft,7.00,8.75,7.00,7.625000,45.0,train
1579,2018-11-26 09:00:00,Lyft,Back Bay,Lyft,7.00,7.00,8.75,7.291667,15.0,train


In [9]:
# KIEM TRA CHONG RO RI: lag1 == target_price cua snapshot lien truoc, TRONG TUNG SPLIT.
# Phai kiem theo split vi 11-12/12 la buffer bi loai -> dong dau series o 13/12 co
# lag1 tro ve ngay buffer (van la QUA KHU that, khong phai ro ri).
keys = ["cab_type","source","destination","name"]
print("[CHONG RO RI] lag1_price == shift(1) trong tung split:")
for sp in ["train","calibration","test"]:
    s = gp[gp.data_split==sp].sort_values(keys+["snapshot"]).copy()
    s["chk"] = s.groupby(keys, observed=True).target_price.shift(1)
    m = s.lag1_price.notna() & s.chk.notna()
    ok = np.allclose(s.lag1_price[m], s.chk[m])
    print(f"  [{sp:12}] -> {ok}   (n={m.sum():,})")

print(f"\n[CHONG RO RI] observation_age > 0 toan bo -> {(gp.observation_age_minutes.dropna()>0).all()}")
print("\n>>> San sang train. Mo model/model_train.ipynb.")

[CHONG RO RI] lag1_price == shift(1) trong tung split:
  [train       ] -> True   (n=288,407)
  [calibration ] -> True   (n=96,833)
  [test        ] -> True   (n=83,817)

[CHONG RO RI] observation_age > 0 toan bo -> True

>>> San sang train. Mo model/model_train.ipynb.
